In [5]:
# Importing necessary libraries

import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import math
from scipy.spatial.transform import Rotation as R
from scipy.linalg import block_diag
import timeit as timee
#import import_ipynb
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
import matplotlib.collections
#import effort_solver_nn_3_triangles_3D_stiffness_information_Classical_FEM_changed as solv
#import Functions_for_material_law as ML
from sklearn.preprocessing  import MinMaxScaler
from datetime import datetime
start_time = datetime.now()
from mpl_toolkits.mplot3d import Axes3D
import sympy as sym
from matplotlib.patches import Rectangle
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Flatten, Lambda, Dense, Conv2D, MaxPool2D, Dropout,LSTM,Embedding,Layer, LSTM, Conv1D, MaxPool1D, TimeDistributed, Input, GRU
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import metrics
from tensorflow.keras.callbacks import Callback
#import plotly.graph_objects as go
import pickle
import multiprocessing
#from numba import njit
from tensorflow.keras.models import Model, load_model
import keras_lmu
from tensorflow.keras.models import Model, load_model
from keras.layers.rnn.dropout_rnn_cell_mixin import DropoutRNNCellMixin
#from keras.layers.rnn.dropout_rnn_cell_mixin import DropoutRNNCellMixin
import decoding
from decoding import DSNN
from Hybrid_Spiking_Neuron import HSNN
import tensorflow_addons as tfa
%matplotlib qt

# 1. Network initialization

In [6]:
units         = 208

state_h  =  tf.zeros(shape=(1,units))
state_c  =  tf.zeros(shape=(1,units))

state_h1 =  tf.zeros(shape=(1,units))
state_c1 =  tf.zeros(shape=(1,units))

state_h2 =  tf.zeros(shape=(1,units))
state_c2 =  tf.zeros(shape=(1,units))

state_h3 =  tf.zeros(shape=(1,208))
state_c3 =  tf.zeros(shape=(1,208))

state_inputs  = [state_h, state_c, state_h1, state_c1, state_h2, state_c2, state_h3, state_c3]

def my_loss(y_true,y_pred):
    
    squared_difference = tf.square(y_true - y_pred)
    
    return tf.reduce_mean(squared_difference, axis=[1, 0])


def scaled_softplus(x, min_val=0.0, max_val=1.0, M=10.0):
    softplus_x = tf.math.softplus(x)
    softplus_M = tf.math.softplus(tf.constant(M, dtype=x.dtype))
    return min_val + (max_val - min_val) * (softplus_x / softplus_M)


def model_klassiche():


    
    
    cell1                           = HSNN(208, thr = 1e-10)
    cell2                           = HSNN(208, thr = 1e-10)
    cell3                           = RSNN(208, thr = 1e-10)
    
    cell4                           = DSNN(208, thr = 1e-10)

    
    
    layer1                          = tf.keras.layers.RNN(cell1, return_sequences=True, return_state=True)
    
    layer2                          = tf.keras.layers.RNN(cell2, return_sequences=True,  return_state=True)
    
    layer3                          = tf.keras.layers.RNN(cell3, return_sequences=True,  return_state=True)
    
    layer4                          = tf.keras.layers.RNN(cell4, return_sequences=True, return_state=True)
    
    layer5                          = tf.keras.layers.BatchNormalization()

    model_state_input_h   = Input(shape=(208,))
    model_state_input_c   = Input(shape=(208,))
    model_state_input_h1  = Input(shape=(208,))
    model_state_input_c1  = Input(shape=(208,))
    model_state_input_h2  = Input(shape=(208,))
    model_state_input_c2  = Input(shape=(208,))
    model_state_input_h3  = Input(shape=(208,))
    model_state_input_c3  = Input(shape=(208,))


    state_inputs = [model_state_input_h, model_state_input_c, model_state_input_h1, model_state_input_c1, model_state_input_h2, model_state_input_c2, model_state_input_h3, model_state_input_c3]

    inputs = Input((None, 12))

    X, h1, c1     = layer1(initial_state=[model_state_input_h, model_state_input_c], inputs=inputs)
    
    X2, h2, c2    = layer2(initial_state=[model_state_input_h1, model_state_input_c1], inputs=(X))
    
    X3, h3, c3    = layer3(initial_state=[model_state_input_h2, model_state_input_c2], inputs=(X2))
    
    V, h4, c4     = layer4(initial_state=[model_state_input_h3, model_state_input_c3],inputs=(X3))

    V             = tf.math.tanh(V)
    
    V             = Dense(240, activation='tanh')(V)
    
    V             = Dense(64, activation='tanh')(V)
    
    V1            = Dense(1, activation='softplus')(V)
    
    
    
    
    model2 = Model(inputs=[inputs]+state_inputs, outputs=[V1, h1, c1, h2, c2, h3, c3, h4, c4])
    
    
    opt = Adam(0.001)
    
    model2.compile(optimizer = opt, loss = my_loss)

    # model2_ = Model(inputs=inputs, outputs=[V1])
    
    
    # opt = Adam(0.001)
    
    # model2_.compile(optimizer = opt, loss = my_loss)

    return model2


def model_klassiche_up():


    
    
    cell1                           = HSNN(208, thr = 1e-10)
    cell2                           = HSNN(208, thr = 1e-10)
    cell3                           = HSNN(208, thr = 1e-10)
    
    cell4                           = DSNN(208, thr=1e-10)

    
    
    layer1                          = tf.keras.layers.RNN(cell1, return_sequences=True, return_state=True)
    
    layer2                          = tf.keras.layers.RNN(cell2, return_sequences=True,  return_state=True)
    
    layer3                          = tf.keras.layers.RNN(cell3, return_sequences=True,  return_state=True)
    
    layer4                          = tf.keras.layers.RNN(cell4, return_sequences=True, return_state=True)
    
    layer5                          = tf.keras.layers.BatchNormalization()

    model_state_input_h   = Input(shape=(208,))
    model_state_input_c   = Input(shape=(208,))
    model_state_input_h1  = Input(shape=(208,))
    model_state_input_c1  = Input(shape=(208,))
    model_state_input_h2  = Input(shape=(208,))
    model_state_input_c2  = Input(shape=(208,))
    model_state_input_h3  = Input(shape=(208,))
    model_state_input_c3  = Input(shape=(208,))
    
    min_val      = Input(shape=(1,))
    max_val      = Input(shape=(1,))
    state_inputs = [model_state_input_h, model_state_input_c, model_state_input_h1, model_state_input_c1, model_state_input_h2, model_state_input_c2, model_state_input_h3, model_state_input_c3]
    range_       = [min_val, max_val]
    


    inputs = Input((None, 12))

    X, h1, c1     = layer1(initial_state=[model_state_input_h, model_state_input_c], inputs=inputs)
    
    X2, h2, c2    = layer2(initial_state=[model_state_input_h1, model_state_input_c1], inputs=(X))
    
    X3, h3, c3    = layer3(initial_state=[model_state_input_h2, model_state_input_c2], inputs=(X2))
    
    V, h4, c4     = layer4(initial_state=[model_state_input_h3, model_state_input_c3],inputs=(X3))

    V             = tf.math.tanh(V)
    
    V             = Dense(240, activation='tanh')(V)
    
    V             = Dense(64, activation='tanh')(V)
    
    V1            = Dense(1, activation='softplus')(V)
    
    V1            = Dense(1)(V)
    V1            = Lambda(lambda x: scaled_softplus(x[0], x[1], x[2]))([V1, min_val, max_val])   
    
    
    
    model2 = Model(inputs=[inputs]+state_inputs+range_, outputs=[V1, h1, c1, h2, c2, h3, c3, h4, c4])
    
    
    opt = Adam(0.003)
    
    model2.compile(optimizer = opt, loss = my_loss)

    # model2_ = Model(inputs=inputs, outputs=[V1])
    
    
    # opt = Adam(0.001)
    
    # model2_.compile(optimizer = opt, loss = my_loss)

    return model2



model = model_klassiche_up()

with open(r"C:\Users\Tandale\sciebo\olddata\PostDoc\Writing\Meta_Learning\Plots\Saved_weights\Version_1\model_weights.pkl", "rb") as f:
    loaded_weights = pickle.load(f)

model.set_weights(loaded_weights)

# model.layers[1].trainable = False
# model.layers[2].trainable = False
# model.layers[3].trainable = False
# #model.layers[4].trainable = False



# 2. Material model parameters and desired functions

In [7]:

def gen_input(stress_dev_trial, stress_back_trial):

    comp1, comp2, comp3, comp4, comp5, comp6                                             = stress_dev_trial[0][0], stress_dev_trial[1][1], stress_dev_trial[2][2], stress_dev_trial[1][2], stress_dev_trial[0][2], stress_dev_trial[0][1]
    compX1, compX2, compX3, compX4, compX5, compX6                                       = stress_back_trial[0][0], stress_back_trial[1][1], stress_back_trial[2][2], stress_back_trial[1][2], stress_back_trial[0][2], stress_back_trial[0][1]
    #comp1, comp2, comp3, comp4, comp5, comp6       = ds
    #compX1, compX2, compX3, compX4, compX5, compX6 = sb
    #int(comp1, comp2, comp3, comp4, comp5, comp6, compX1, compX2, compX3, compX4, compX5, compX6)
    x_batch_train                                  = np.array([[comp1, comp2, comp3, comp4, comp5, comp6, compX1, compX2, compX3, compX4, compX5, compX6]])/100
    x_batch_train                                  = tf.cast(abs(x_batch_train), dtype = tf.float32)
    x_batch_train                                  = tf.reshape(x_batch_train, shape=(1,1,12))
    
    return x_batch_train


def tens_to_voigt(tens):
    
    stress    = np.zeros(shape=(6,))
    
    stress[0] = tens[0][0]
    stress[1] = tens[1][1]
    stress[2] = tens[2][2]
    stress[3] = tens[1][2]
    stress[4] = tens[0][2]
    stress[5] = tens[0][1]
    
    return stress


def voigt_to_tens(stress):
    
    tens = np.zeros(shape=(3,3))
    
    
    tens[0][0] = stress[0]
    tens[1][1] = stress[1]
    tens[2][2] = stress[2]
    tens[0][1] = stress[5]
    tens[0][2] = stress[4]
    tens[1][0] = stress[5]
    tens[1][2] = stress[3]
    tens[2][0] = stress[4]
    tens[2][1] = stress[3]
    
    
    return tens


def props_update(mat_params, i,l,j,k, pois):
    
    return (mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]*pois)/((1 + pois)*(1 - 2*pois)), 0.5*mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]/(1 + pois)

def split(tens):
   
    I      = np.eye(3,3)
    
    aa     = voigt_to_tens(tens)
    
    shydro = (1/3*(aa.trace()))
    
    return aa - shydro*I, shydro*I


#......................................................................................................................................................

wahl                        = 0
height                      = 1
thick                       = 1
number_of_elements          = 1
DOF_per_node                = 5
theta                       = 0
pois                        = 0.32
nn                          = {}
nodes                       = {}
ax                          = {}
result                      = []
stiffness                   = []
N                           = 2.79
dummy                       = [] # This variable is used so that while doing df/dx we can find loop it for calculating
count                       = 0
NEWTON                      = 300
nodes_per_element           = 9
timeincrement               = 1e-4





# Initializing all the material properties

mat_params    = {}
I             = np.eye(3,3)
units         = 208


for i in range(1):
    
    for l in range(1):
    
        for j in range(1):

            for k in range(1):


                mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = 0 
                mat_params['stress_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['flow'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                              = np.zeros(shape=(3,3))
                mat_params['eps_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = np.zeros(shape=(3,3))
                mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                               = np.zeros(shape=(3,3))
                mat_params['stress_n'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = np.zeros(shape=(6,))
                mat_params['stress_yield'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                      = 110
                mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                 = 67400
                mat_params['delta_eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                         = np.zeros(shape=(6,))
                mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                     = np.zeros(shape=(6,))
                #mat_params['strain_n_1']                                                   = np.zeros(shape=(6,))
                mat_params['lam'+str(i+1)+str(l+1)+str(j+1)+str(k+1)], mat_params['mu'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]   = props_update(mat_params, i,l,j,k, pois) 
                mat_params['pois'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                              = pois
                mat_params['stress_n-1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = np.zeros(shape=(6,))
                mat_params['eps_vec'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                           = np.zeros(shape=(6,))
                aaa, bbb = mat_params['lam'+str(i+1)+str(l+1)+str(j+1)+str(k+1)], mat_params['mu'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]
                aa = 2*bbb + aaa
                mat_params['stress_K'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                          = 3.42
                mat_params['abs_stress'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = []
                mat_params['stress_back'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = np.zeros(shape=(3,3))
                mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = np.zeros(shape=(3,3))
                mat_params['abs_strain'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = []
                mat_params['C'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                 = np.array([[aa, aaa, aaa, 0, 0, 0], 
                           [aaa, aa, aaa, 0, 0, 0],
                           [aaa, aaa, aa, 0, 0, 0],
                          [0, 0, 0, bbb,  0,  0],
                          [0, 0, 0, 0,   bbb, 0, ],
                          [0, 0, 0, 0, 0, bbb]])
                mat_params['C_evp_fourth'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                               = np.zeros(shape=(3,3,3,3))
                mat_params['C_evp'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                      = mat_params['C'+str(i+1)+str(l+1)+str(j+1)+str(k+1)].copy()

                mat_params['C_ep'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                                                    = np.array([[aa, aaa, aaa, 0, 0, 0], 
                           [aaa, aa, aaa, 0, 0, 0],
                           [aaa, aaa, aa, 0, 0, 0],
                          [0, 0, 0, bbb,  0,  0],
                          [0, 0, 0, 0,   bbb, 0, ],
                          [0, 0, 0, 0, 0, bbb]])

                mat_params['EQPLAS'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = np.sqrt((np.tensordot(mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)],mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]))*(2/3))
                K                                                                          = mat_params['E'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]/(1 - 2*pois)
                mat_params['Delta_d'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = 0
                mat_params['Delta_pl'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                 = 0
                mat_params['a_unit'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = 3100
                mat_params['s_unit'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                   = 110
                mat_params['Delta_epsilon_p'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]          = []
                mat_params['input'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                    = []
                mat_params['R'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                        = 0.0
                mat_params['b1'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = 0.
                mat_params['b2'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                       = 0.0
                mat_params['state_h_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = tf.zeros(shape=(1,units))
                mat_params['state_c_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  =  tf.zeros(shape=(1,units)) 
                mat_params['state_h1_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = tf.zeros(shape=(1,units))
                mat_params['state_c1_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  =  tf.zeros(shape=(1,units)) 
                mat_params['state_h2_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = tf.zeros(shape=(1,units))
                mat_params['state_c2_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  =  tf.zeros(shape=(1,units)) 
                mat_params['state_h3_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  = tf.zeros(shape=(1,units))
                mat_params['state_c3_'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]                  =  tf.zeros(shape=(1,units)) 
                mat_params['seq_delta_eps'+str(i+1)+str(l+1)+str(j+1)+str(k+1)]              = []
                
            mat_params['stiff_point'+str(i+1)+str(l+1)+str(j+1)]                           = np.zeros(shape=(DOF_per_node*nodes_per_element, DOF_per_node*nodes_per_element))     
    
    ax['element'+str(i+1)]                                                                 = np.zeros(shape=(DOF_per_node*nodes_per_element ,1))
    nn['element_force'+str(i+1)]                                              = []
    nn['element_disp'+str(i+1)]                                               = []
    nn['element_stiff'+str(i+1)]                                              = []





# 3. Loesung Verfahren

In [8]:
def train_LSTM_static_KI(x_batch_train, stress_dev_trial, stress_back_trial, N_dirr, timeincrement, mat_params, N, state_inputs, model, opt,outer_step_size,i,l,m,n, stress_eqv_trial):
    
    
#     BATCH_SIZE = 64
#     EPOCHS = 100
#     INIT_LR = 1e-3
#     MAX_LR = 2e-3


#     steps_per_epoch = 10000 // BATCH_SIZE
#     clr             = tfa.optimizers.CyclicalLearningRate(initial_learning_rate=INIT_LR,
#         maximal_learning_rate=MAX_LR,
#         scale_fn=lambda x: 1/(2.**(x-1)),
#         step_size=2 * steps_per_epoch)

    #opt                         =   Adam(0.001)
    #weights_before              =   model.get_weights()

    #min_val  = 0

    

    #if ((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])<0:
        #print(i, l, m, n, stress_eqv_trial, mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], N)
    
    # Delta_pl_U1 = np.maximum(0,(((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))*timeincrement)
    # #Delta_pl_U1 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial-mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(m+1)+str(n+1)]))
    # mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    # stress_dev_trial_n  =   mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
    # stress_eqv_trial_n  = np.sqrt(1.5*(np.tensordot(stress_dev_trial_n - stress_back_trial, stress_dev_trial-stress_back_trial)))
    # Delta_pl_U2 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial_n-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))



    # if Delta_pl_U2>Delta_pl_U1==0:

    #     max_val = Delta_pl_U2
    #     #print('Delta_pl_U1 is chosen')

    # else:
    #     max_val = Delta_pl_U1 
        #print('Delta_pl_U2 is chosen')

    #print(f'min_val:{min_val} max_val: {max_val}')
    #range_ = [tf.constant(min_val, dtype=tf.float32), tf.constant(max_val, dtype=tf.float32)]
    
    with tf.GradientTape(persistent=True) as tt:

        F_n, h, c, h1, c1, h2, c2, h3, c3    = model([x_batch_train]+state_inputs)  
        tt.watch(model.trainable_weights)
        
        h_N                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*((F_n)[0][0][0]/timeincrement)**(1/N) + (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(F_n)[0][0][0])/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(F_n)[0][0][0])
        Z_N                      =         stress_dev_trial - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 110*(F_n)[0][0][0]))
        alpha_N                  =         1 + (3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (3100/(1 + 110*(F_n)[0][0][0])))*((F_n)[0][0][0]/h_N)
        J_N                      =         np.sqrt(1.5*np.tensordot(Z_N, Z_N))
        RHS_N                    =         (1/alpha_N)*(J_N) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*((F_n[0][0][0])/timeincrement)**(1/N) - (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(F_n[0][0][0]))/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(F_n[0][0][0]))

        loss_value   = tf.reduce_mean(tf.math.pow(RHS_N,2)) + tf.math.pow((1-tf.math.sign(F_n[0][0][0]))*tf.math.abs(F_n[0][0][0]), 2)*0
        

    grads_batch      = tt.gradient(loss_value, model.trainable_weights)
    
    
    opt.apply_gradients(zip(grads_batch,model.trainable_weights))
    
    #weights_after    = model.get_weights()
    
    #model.set_weights([ weights_before[i] + (weights_after[i] - weights_before[i]) * outer_step_size 
    #    for i in range(len(model.weights))])
    
    
    #print(F_n)

    #print(f'min_val:{min_val} max_val: {max_val} EQPLAS {F_n[0][0][0].numpy()}')
    
    return loss_value, (F_n), h, c, (((Z_N/h_N)).numpy()), h1, c1, h2, c2, h3, c3, h_N, Z_N, alpha_N, J_N


def train_LSTM_static_KI_(x_batch_train, stress_dev_trial, stress_back_trial, N_dirr, timeincrement, mat_params, N, state_inputs, model, opt,outer_step_size,i,l,m,n, stress_eqv_trial):
    
    
#     BATCH_SIZE = 64
#     EPOCHS = 100
#     INIT_LR = 1e-3
#     MAX_LR = 2e-3


#     steps_per_epoch = 10000 // BATCH_SIZE
#     clr             = tfa.optimizers.CyclicalLearningRate(initial_learning_rate=INIT_LR,
#         maximal_learning_rate=MAX_LR,
#         scale_fn=lambda x: 1/(2.**(x-1)),
#         step_size=2 * steps_per_epoch)

    #opt                         =   Adam(0.001)
    #weights_before              =   model.get_weights()

    min_val  = 0

    

    #if ((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])<0:
        #print(i, l, m, n, stress_eqv_trial, mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], N)
    
    Delta_pl_U1 = np.maximum(0,(((stress_eqv_trial - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])**(N))*timeincrement)
    #Delta_pl_U1 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial-mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(m+1)+str(n+1)]))
    mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
    stress_dev_trial_n  =   mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
    stress_eqv_trial_n  = np.sqrt(1.5*(np.tensordot(stress_dev_trial_n - stress_back_trial, stress_dev_trial-stress_back_trial)))
    Delta_pl_U2 = (np.sqrt((2/3)*(np.tensordot(voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]), voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])))) + (stress_eqv_trial_n-mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(mat_params['E'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))



    if Delta_pl_U2>Delta_pl_U1==0:

        max_val = Delta_pl_U2
        #print('Delta_pl_U1 is chosen')

    else:
        max_val = Delta_pl_U1 
        #print('Delta_pl_U2 is chosen')

    #print(f'min_val:{min_val} max_val: {max_val}')
    range_ = [tf.constant(min_val, dtype=tf.float32), tf.constant(max_val, dtype=tf.float32)]
    
    with tf.GradientTape(persistent=True) as tt:

        F_n, h, c, h1, c1, h2, c2, h3, c3    = model([x_batch_train]+state_inputs+range_)  
        tt.watch(model.trainable_weights)
        
        h_N                      =         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*((F_n)[0][0][0]/timeincrement)**(1/N) + (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(F_n)[0][0][0])/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(F_n)[0][0][0])
        Z_N                      =         stress_dev_trial - ((mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])/(1 + 110*(F_n)[0][0][0]))
        alpha_N                  =         1 + (3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (3100/(1 + 110*(F_n)[0][0][0])))*((F_n)[0][0][0]/h_N)
        J_N                      =         np.sqrt(1.5*np.tensordot(Z_N, Z_N))
        RHS_N                    =         (1/alpha_N)*(J_N) - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['stress_K'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*((F_n[0][0][0])/timeincrement)**(1/N) - (mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] - mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*mat_params['b2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(F_n[0][0][0]))/(1 + mat_params['b1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(F_n[0][0][0]))

        loss_value   = tf.reduce_mean(tf.math.pow(RHS_N,2)) + tf.math.pow((1-tf.math.sign(F_n[0][0][0]))*tf.math.abs(F_n[0][0][0]), 2)*0
        

    grads_batch      = tt.gradient(loss_value, model.trainable_weights)
    
    
    opt.apply_gradients(zip(grads_batch,model.trainable_weights))
    
    #weights_after    = model.get_weights()
    
    #model.set_weights([ weights_before[i] + (weights_after[i] - weights_before[i]) * outer_step_size 
    #    for i in range(len(model.weights))])
    
    
    #print(F_n)

    #print(f'min_val:{min_val} max_val: {max_val} EQPLAS {F_n[0][0][0].numpy()}')
    
    return loss_value, (F_n), h, c, (((Z_N/h_N)).numpy()), h1, c1, h2, c2, h3, c3, h_N, Z_N, alpha_N, J_N






# Test von Integration Schema

In [9]:
l = 0
m = 0
n = 0
random   = 0
Delta_pl = 0
wahl    = 0
deqpl   = []
deqpl1           = []
deqpl2           = []
eps33_ana        = []
eps33_ana_pl     = []
count_tr         = 0
sequence_tracker = 0
trial_stress = []
back_stress  = []
number_of_iteration = 0
start_time = datetime.now()
iterations    = []

increment_s = [18]

sign   =  [1]

for ll in range(len(sign)):
    
    print('......................................................................................................')
    print('Load Sequence '+str(l+1)+' The sign is '+str(sign[l]))
    print('......................................................................................................')

    mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = np.array([ 1.39385621e-04, -4.35790497e-07, -6.53881554e-05, -1.20143493e-06,
       -6.00364241e-06,  8.29015628e-07])
     
    for t in range(increment_s[ll]):

        mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = np.array([ 1.39385621e-04, -4.35790497e-07, -6.53881554e-05, -1.20143493e-06,
       -6.00364241e-06,  8.29015628e-07])
        for l in range(1):
        
            for m in range(1):
    
                for n in range(1):
    
                    mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]      = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                    sig_tr_i_1                                                        = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    mat_params['stress_back_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                    #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) 
                    #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  
                    stress_back_trial                                                 = mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                    #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    stress_dev_trial,_                                                = split(sig_tr_i_1)
                    #if i==0 and m==0 and n==0:
                    #    print(mat_params['stress_n111'])
                    stress_eqv_trial                                                  = np.sqrt(1.5*(np.tensordot(stress_dev_trial-stress_back_trial, stress_dev_trial-stress_back_trial)))    
                    N_dirr                                                            = ((stress_dev_trial - stress_back_trial)/stress_eqv_trial)
    
    
                    # Write a function to give the current yield stress
                    #mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = Uhard(mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    #print(mat_params)
    
                    if stress_eqv_trial > abs(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]):
    
                        print("Viskoplastizitat ist angreteten")
                        state_inputs = [mat_params['state_h_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_h1_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c1_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_h2_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c2_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_h3_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c3_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]]
                        
                        for p in range(100):
    
                            if not(p==0):
    
                                #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps_cor'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                #mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])  
                                #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = voigt_to_tens(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                #stress_dev_trial                                              = mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]-I*(1/3*mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].trace())
                                #if i==0 and m==0 and n==0:
                                
                                #    print(mat_params['stress_n111'])
                                sig_tr_i_1                                                    = mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                                stress_dev_trial,_                                            = split(sig_tr_i_1)
                                stress_eqv_trial                                              = np.sqrt(1.5*(np.tensordot(stress_dev_trial-stress_back_trial, stress_dev_trial-stress_back_trial)))    
                                sigma_33_trial                                                = sig_tr_i_1[2]
                                #Delta_pl, alpha_N, h_N, mat_params,N_plas                     = implicit_integration_vp_gnl(mat_params, i, gauss_points, section_points, N, timeincrement, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n)
                                iso                                                           = np.array([[mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]]])
                                x_batch_train                                                 = gen_input(stress_dev_trial, stress_back_trial)
                                
                                #..................... Introduction to NN enhanced integration scheme ......................
                                state_inputs  =  [mat_params['state_h_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_h1_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c1_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_h2_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_c2_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_h3_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_c3_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]]
                                #model.set_weights(loaded_weights)
                                #print(x_batch_train)
                                opt = Adam(0.003)
                                for dt in range(1000):
    
                                    loss_value, F_n, h_, c_, N_plas, h_1, c_1, h_2, c_2, h_3, c_3, h_N, Z_N, alpha_N, J_N             = train_LSTM_static_KI_(x_batch_train, tf.convert_to_tensor(stress_dev_trial, dtype=tf.float32), tf.convert_to_tensor(stress_back_trial, dtype=tf.float32), tf.convert_to_tensor(N_dirr, dtype=tf.float32), timeincrement, mat_params, N, state_inputs, model, opt, 0.01, i ,l ,m ,n , stress_eqv_trial)
    
                                    #print(loss_value)
    
    
                                    if loss_value<1e-4:
    
                                        Delta_pl = F_n[0][0][0].numpy()
                                        #print('NN converges well')
    
                                        break
    
                                
                                
                                Delta_pl = F_n[0][0][0].numpy()
                                
                                #....................................................................................................
                                
                                sigma_i_1                                                     = sigma_33_trial - ((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl))*N_plas[2][2]
                                delta_eps33_i_1                                               = delta_eps33_i_minus_1 -(((sigma_i)*(delta_eps33_i - delta_eps33_i_minus_1))/(1e-7 + sigma_i_1 - sigma_i))
                                mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = delta_eps33_i_1
                                delta_eps33_i_minus_1 =  delta_eps33_i
                                delta_eps33_i         =  delta_eps33_i_1
                                sigma_i               =  sigma_i_1
                                
                                
                                if abs(delta_eps33_i - delta_eps33_i_minus_1)/abs(delta_eps33_i_1) < 1e-5:
                                    print('Die Konvergenz erfolgt nach '+str(p+1)+' Iterationen für '+str(t+1)+' inkrement')   
                                    #mat_params['input'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append([ML.tens_to_voigt(stress_dev_trial), ML.tens_to_voigt(stress_back_trial)])
                                    #mat_params['Delta_epsilon_p'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(Delta_pl)
                                    break
                                #print('Trial_Stress '+str(sigma_33_trial),'Sigma_i_1 ' +str(sigma_i_1), 'Equivalent Plastic Strain: '+str(Delta_pl))
                                
                                
                                
                                
                            elif p==0:
                        
                                sigma_33_trial        = sig_tr_i_1[2]
                                #mat_params, Delta_pl  = implicit_integration(mat_params, stress_eqv_trial,i,l,m,n)
                                #Delta_pl, alpha_N, h_N, mat_params, N_plas  = implicit_integration_vp_gnl(mat_params, i, gauss_points, section_points, N, timeincrement, wahl, stress_eqv_trial,stress_back_trial, stress_dev_trial,l,m,n)
                                
                                #..................... Introduction to NN enhanced integration scheme ......................
                                
                                iso                                                           = np.array([[mat_params['R'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]]])
                                x_batch_train                                                 = gen_input(stress_dev_trial, stress_back_trial)
                                #state_inputs  =  [mat_params['state_h_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_h_1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c_1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_h_2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_c_2'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_h_3'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_c_3'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]]
                                
                                state_inputs  =  [mat_params['state_h_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_h1_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_c1_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['state_h2_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_c2_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_h3_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)],mat_params['state_c3_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]]
                                opt           =  Adam(0.003)
                                #model.set_weights(loaded_weights)
                                for dt in range(5000):
    
                                    loss_value, F_n, h_, c_, N_plas, h_1, c_1, h_2, c_2, h_3, c_3, h_N, Z_N, alpha_N, J_N             = train_LSTM_static_KI_(x_batch_train, tf.convert_to_tensor(stress_dev_trial, dtype=tf.float32), tf.convert_to_tensor(stress_back_trial, dtype=tf.float32), tf.convert_to_tensor(N_dirr, dtype=tf.float32), timeincrement, mat_params, N, state_inputs, model, opt,0.01,i,l,m,n, stress_eqv_trial)
    
                                    #print(loss_value)
    
    
                                    if loss_value<1e-4:
    
                                        #Delta_pl = F_n[0][0][0].numpy()
                                        #print('NN converges well')
    
                                        break
    
                                
                                
                                Delta_pl = F_n[0][0][0].numpy()
              
                                
                                #....................................................................................................                            
                                
                                sigma_i_1             = sigma_33_trial - ((3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl))*N_plas[2][2]
                                sigma_i               = sigma_i_1
                                delta_eps33_i_minus_1 = mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] 
                                mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2] = -(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][0] + mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][1])
                                delta_eps33_i         = mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2]
                                
                                #print(delta_eps33_i, delta_eps33_i_minus_1, Delta_pl)
                        #Delta_pl    = 0
                        #mat_params['flag'+str(i+1)]+=1
                        #print(mat_params['flag'+str(i+1)])
                        #delta_ep    = 0
    
    #                     for j in range(1000):
    
    
    #                         RHS                              = stress_eqv_trial  - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] -  3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl))
    #                         #a                                = RHS/(mat_params['H'+str(i+1)] + (mat_params['E'+str(i+1)]))
    #                         #print(mat_params['H'+str(i+1)])
    #                         Delta_pl                         = Delta_pl +  (RHS/(mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + 3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
    #                         #delta_ep                         = delta_ep + (Delta_pl)
    #                         #print(Delta_pl)
    #                         #mat_params['EQPLAS'+str(i+1)]    = mat_params['EQPLAS'+str(i+1)] + Delta_pl
    
    #                         mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['H'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = Uhard(mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]+Delta_pl)
    #                         #print(mat_params['H'+str(i+1)])
    #                         #print(Delta_pl, mat_params['EQPLAS'+str(i+1)], RHS)
    #                         #mat_params['C'+str(i+1)][0][0]                                = (mat_params['E'+str(i+1)]*mat_params['H'+str(i+1)])/(mat_params['E'+str(i+1)] + mat_params['H'+str(i+1)])
    #                         #print(RHS, mat_params['stress_yield'+str(i+1)])
    #                         if abs(stress_eqv_trial  - mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] -  3*(mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*(Delta_pl)))<1e-5:
    #                             #print(j+1, stress_eqv_trial  - mat_params['stress_yield'+str(i+1)] -  (3*mat_params['mu'+str(i+1)]*(Delta_pl)))
    #                             #print('....Converged...... in '+str(j+1)+' iterations')
    #                             #print(Delta_pl)
    #                             break
    
                        # Now we move ahead with updating the stress, elastic and plastic strains
                        #print('...............out from the loop.......')
    
                        #if mat_params['flag'+str(i+1)]==0:
                        #    mat_params['stress_tens'+str(i+1)]        = ((stress_dev_trial/(stress_eqv_trial))*mat_params['stress_yield'+str(i+1)]) + I*(ss) 
                        #    print(mat_params['stress_tens'+str(i+1)])
                        #    mat_params['flag'+str(i+1)]               +=1
                        #    mat_params['stress_tens'+str(i+1)]        = ((stress_dev_trial/(stress_eqv_trial))*mat_params['stress_yield'+str(i+1)]) + I*(ss)  + np.array([[4.2,0,0],[0,0,0],[0,0,0]])
                        #    print(mat_params['stress_tens'+str(i+1)])
    
                        #else:
                        #if i==0 and m==0 and n==0:
    
                        #    print((mat_params['stress_n111'] - ML.tens_to_voigt(2*mat_params['mu111']*(1.5)*Delta_pl*(stress_dev_trial/(stress_eqv_trial))))[0])
                        #mat_params['stress_yield'+str(i+1)],_ = Uhard(mat_params['EQPLAS'+str(i+1)]+Delta_pl)
                        #dummy_sph                             = (stress_dev_trial/(stress_eqv_trial))
                        #mat_params['stress_yield'+str(i+1)],_ = Uhard(mat_params['EQPLAS'+str(i+1)]+Delta_pl)
                        #mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]    = ((stress_dev_trial/(stress_eqv_trial))*(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])) + I*(mat_params['stress_yield'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*dummy_sph[0][0]/(3*abs(dummy_sph[0][0])))
                        #mat_params['stress_tens'+str(i+1)]    = ((stress_dev_trial/(stress_eqv_trial))*(mat_params['stress_yield'+str(i+1)])) + I*(mat_params['stress_yield'+str(i+1)]/3)
                        #print(((stress_dev_trial/(stress_eqv_trial))*mat_params['stress_yield'+str(i+1)]))
    
                        #print('.............................')
                        #print()
    
                        mat_params['state_h_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]          = h_
                        mat_params['state_c_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]          = c_
                        
                        mat_params['state_h1_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]         = h_1
                        mat_params['state_c1_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]         = c_1
                        
                        mat_params['state_h2_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]          = h_2
                        mat_params['state_c2_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]          = c_2
                        
                        mat_params['state_h3_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]          = h_3
                        mat_params['state_c3_'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]          = c_3
                        
                        
                        #mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]  = ML.tens_to_voigt(mat_params['stress_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        a1                                        = tens_to_voigt(mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        b1                                        = tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        c1                                        = tens_to_voigt(N_plas)
    
                        for j in range(3):
    
                            a1[j]                                 = a1[j]  +  (1.5*(Delta_pl)*(c1[j]/1))
                            b1[j]                                 = b1[j]  -  (1.5*(Delta_pl)*(c1[j]/1))
    
                        for j in range(3,6):
    
                            a1[j]                                 = a1[j]  +  (3*(Delta_pl)*(c1[j]/1))
                            b1[j]                                 = b1[j]  -  (3*(Delta_pl)*(c1[j]/1))
    
    
                        #mat_params['stress_n'+str(i+1)]           = mat_params['stress_n'+str(i+1)] + mat_params['C_ep'+str(i+1)].dot(mat_params['delta_eps'+str(i+1)]-1.5*(Delta_pl)*(c1/stress_eqv_trial))
    
                        mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = voigt_to_tens(a1)
                        mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = voigt_to_tens(b1)
                        mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]             = mat_params['EQPLAS'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + Delta_pl
                        mat_params['flow'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]               = tens_to_voigt(N_plas)
                        mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]) - (Delta_pl*mat_params['flow'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])*3*mat_params['mu'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['stress_n-1'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]                                          
                        #mat_params['flow'+str(i+1)]               = mat_params['flow'+str(i+1)].reshape(6,1)
                        mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]       = (mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + (mat_params['a_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl*N_plas))/(1 + mat_params['s_unit'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]*Delta_pl)
                        #print(mat_params['EQPLAS'+str(i+1)], Delta_pl)
                        # Updating the material jacobian with a function
                        mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]   = stress_eqv_trial
                        #mat_params    = material_jacobian_evp(mat_params, i, N, N_plas, I, l, m, n, Delta_pl, J_N, index, stress_dev_trial, timeincrement, Z_N, h_N, alpha_N)
                        mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                        mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append((mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                        mat_params['Delta_pl'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = Delta_pl
                        #print(mat_params['EQPLAS'+str(i+1)+str(m+1)+str(n+1)])
                        mat_params['stress_back'+str(i+1)+str(l+1)+str(m+1)+str(n+1)][2][2]  = 0
                        #print('Stress_yield through material law:',mat_params['stress_yield'+str(i+1)+str(m+1)+str(n+1)])
    
                        #print('....................................')
                        #print(mat_params['EQPLAS1'])
    
                    else:
    
                        mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]        = mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + voigt_to_tens(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]           = mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]  + mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].dot(mat_params['delta_eps'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                        mat_params['stress_eqv_trial'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]   = stress_eqv_trial
                        mat_params['C_ep'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]               = mat_params['C'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].copy()
                        mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(tens_to_voigt(mat_params['eps_el_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)] + mat_params['eps_pl_tens'+str(i+1)+str(l+1)+str(m+1)+str(n+1)]))
                        mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)].append(mat_params['stress_n'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])
                    

......................................................................................................
Load Sequence 1 The sign is 1
......................................................................................................
Viskoplastizitat ist angreteten
Die Konvergenz erfolgt nach 6 Iterationen für 12 inkrement
Viskoplastizitat ist angreteten
Die Konvergenz erfolgt nach 8 Iterationen für 13 inkrement
Viskoplastizitat ist angreteten
Die Konvergenz erfolgt nach 7 Iterationen für 14 inkrement
Viskoplastizitat ist angreteten
Die Konvergenz erfolgt nach 17 Iterationen für 15 inkrement
Viskoplastizitat ist angreteten
Die Konvergenz erfolgt nach 16 Iterationen für 16 inkrement
Viskoplastizitat ist angreteten
Die Konvergenz erfolgt nach 16 Iterationen für 17 inkrement
Viskoplastizitat ist angreteten
Die Konvergenz erfolgt nach 6 Iterationen für 18 inkrement


In [10]:
plt.plot(mat_params['abs_strain'+str(i+1)+str(l+1)+str(m+1)+str(n+1)], mat_params['abs_stress'+str(i+1)+str(l+1)+str(m+1)+str(n+1)])